# Phase 4.2 — Leakage-Controlled Holdout Audit

This reproducible audit checks split size, spatial coverage, nearest-pose separation and representative RGB content. All split decisions come from production APIs.

## Context & Methods

The holdout prevents validation and its two nearest pose neighbors from entering training. Geometry remains conditioned on the provided shared COLMAP reconstruction; RGB sampling is isolated by exact image names.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'src').is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

from bts_nvs.data.dataset import SceneDataset
from bts_nvs.data.holdout import build_pose_holdout, manifest_pose_distance_matrix
from bts_nvs.data.manifest import load_scene_manifest

SCENE_ID = 'HCM0181'
SCENE_ROOT = REPO_ROOT / 'data' / 'phase1' / 'public_set' / SCENE_ID
MANIFEST_JSON = REPO_ROOT / 'runs' / 'manifests' / SCENE_ID / 'manifest.json'
assert SCENE_ROOT.is_dir(), f'Missing scene: {SCENE_ROOT}'
assert MANIFEST_JSON.is_file(), f'Missing manifest: {MANIFEST_JSON}'

COLORS = {'train': '#2563EB', 'guard': '#D97706', 'validation': '#DB2777'}
MARKERS = {'train': 'o', 'guard': '^', 'validation': 's'}
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

## Results

In [ ]:
manifest = load_scene_manifest(MANIFEST_JSON, SCENE_ROOT)
split = build_pose_holdout(manifest)
groups = {
    'train': set(split.train_image_names),
    'guard': set(split.guard_image_names),
    'validation': set(split.validation_image_names),
}
assert set.union(*groups.values()) == set(manifest.train_image_names)
assert all(groups[a].isdisjoint(groups[b]) for a, b in [('train', 'guard'), ('train', 'validation'), ('guard', 'validation')])

fig, axes = plt.subplots(1, 4, figsize=(12, 2.2))
cards = [
    ('Physical', len(manifest.train_image_names), '#111827'),
    ('Internal train', len(groups['train']), COLORS['train']),
    ('Guard', len(groups['guard']), COLORS['guard']),
    ('Validation', len(groups['validation']), COLORS['validation']),
]
for axis, (label, value, color) in zip(axes, cards):
    axis.axis('off')
    axis.text(0.5, 0.62, str(value), ha='center', va='center', fontsize=25, fontweight='bold', color=color)
    axis.text(0.5, 0.25, label, ha='center', va='center', fontsize=10, color='#4B5563')
fig.suptitle(f'{SCENE_ID} holdout composition', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Camera coverage

In [ ]:
name_to_index = {name: index for index, name in enumerate(manifest.train_image_names)}
centers = manifest.train_camera_to_world[:, :3, 3]
transform = manifest.normalization_transform
normalized_centers = centers @ transform[:3, :3].T + transform[:3, 3]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for label in ('train', 'guard', 'validation'):
    indices = [name_to_index[name] for name in groups[label]]
    points = normalized_centers[indices]
    axes[0].scatter(points[:, 0], points[:, 1], s=24, alpha=0.82, c=COLORS[label], marker=MARKERS[label], label=label.title())
    axes[1].scatter(points[:, 0], points[:, 2], s=24, alpha=0.82, c=COLORS[label], marker=MARKERS[label], label=label.title())
axes[0].set(title='Top view', xlabel='Normalized X', ylabel='Normalized Y', aspect='equal')
axes[1].set(title='Side view', xlabel='Normalized X', ylabel='Normalized Z')
for axis in axes:
    axis.grid(alpha=0.18)
    axis.legend(frameon=False)
fig.suptitle('Camera-center coverage by split', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Nearest-pose distance

In [ ]:
ordered_names, pose_distances = manifest_pose_distance_matrix(manifest)
distance_work = pose_distances.copy()
np.fill_diagonal(distance_work, np.inf)
nearest = distance_work.min(axis=1)
labels = np.asarray([next(label for label, names in groups.items() if name in names) for name in ordered_names])
bins = np.linspace(0.0, float(np.percentile(nearest, 99)) * 1.05, 18)
fig, axis = plt.subplots(figsize=(9, 4.2))
for label in ('train', 'guard', 'validation'):
    axis.hist(nearest[labels == label], bins=bins, alpha=0.55, color=COLORS[label], label=label.title())
axis.set(title='Nearest-camera pose distance', xlabel='D(i, nearest j)', ylabel='Camera count')
axis.grid(axis='y', alpha=0.18)
axis.legend(frameon=False)
plt.tight_layout()
plt.show()

### Validation contact sheet

In [ ]:
preview_names = split.validation_image_names[:12]
preview = SceneDataset(manifest, SCENE_ROOT, image_names=preview_names, resize=(240, 180))
fig, axes = plt.subplots(3, 4, figsize=(13, 8))
for axis, sample in zip(axes.flat, preview):
    axis.imshow(sample.image)
    axis.set_title(sample.image_name, fontsize=8)
    axis.axis('off')
for axis in axes.flat[len(preview):]:
    axis.axis('off')
fig.suptitle('Validation views selected by pose FPS', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Takeaways

In [ ]:
display(Markdown(
    f'**{SCENE_ID} passes the Phase 4.2 split gate:** '
    f'{len(groups["train"])} train, {len(groups["guard"])} guard and '
    f'{len(groups["validation"])} validation cameras. The partitions are disjoint, '
    'cover every physical train image, and exclude official test names.'
))